In [6]:
import plotly.graph_objects as go
import pandas as pd

# 1. Load the Datasets
df = pd.read_excel("Content Table.xlsx", sheet_name="Content TABLE ")
df_dam = pd.read_excel("Content Table.xlsx", sheet_name="DAM Station ELEV(GIS)")

# 2. Extract Arrays
elevation = df['Elevation (m)'].values
volume = df['Volume (MCM)'].values

# 3. Create Figure
fig = go.Figure()

# Add the Capacity Curve (Drafting style: thick black line, cross markers)
fig.add_trace(go.Scatter(
    x=volume,
    y=elevation,
    mode='lines+markers',
    name='Storage Capacity',
    line=dict(color='black', width=3, dash='solid'),
    marker=dict(symbol='cross', size=6, color='black', line=dict(width=1))
))

# 4. Identify Key Levels
levels = []
for idx, row in df.dropna(subset=['Remarks']).iterrows():
    levels.append((row['Elevation (m)'], row['Remarks']))

# Add TBL from DAM Station GIS file
tbl_elev = df_dam['Crest Elevation (m)'].max()
levels.append((tbl_elev, 'Top of Bund Level (TBL)'))

# 5. Add Horizontal Lines & CAD-style Annotations
for el, name in levels:
    fig.add_hline(
        y=el, line_dash="dashdot", line_color="red", line_width=1.5,
        annotation_text=f" {name} [EL {el:.2f} m] ",
        annotation_position="top left",
        annotation_font=dict(family="Courier New, monospace", size=12, color="black"),
        annotation_bgcolor="white",
        annotation_bordercolor="black",
        annotation_borderwidth=1
    )

# 6. Apply "Engineering Graph Paper" Formatting
fig.update_layout(
    title=dict(
        text='<b>JAWALGAON DAM: ELEVATION-CAPACITY CURVE</b><br><sup>DRAWING NO: 01 | SCALE: AS SHOWN</sup>',
        font=dict(family="Courier New, monospace", size=18, color="black"),
        x=0.5,
        xanchor='center'
    ),
    xaxis_title='<b>STORAGE VOLUME (MCM)</b>',
    yaxis_title='<b>ELEVATION (m)</b>',
    plot_bgcolor='rgb(255, 255, 255)', # Pure white paper background
    paper_bgcolor='rgb(245, 245, 245)', # Slight grey surround to mimic a drawing sheet
    font=dict(family="Courier New, monospace", size=12, color="black"),

    # Configure X-Axis (Volume) - Millimeter Graph Paper Look
    xaxis=dict(
        showgrid=True,
        gridcolor='rgb(160, 190, 160)', # Major grid: drafting green
        gridwidth=1.5,
        dtick=10,
        minor=dict(
            dtick=2,
            showgrid=True,
            gridcolor='rgb(210, 230, 210)', # Minor grid: faint green
            gridwidth=0.5
        ),
        zeroline=True,
        zerolinecolor='black',
        zerolinewidth=2,
        showline=True, linewidth=2, linecolor='black', mirror=True,
        ticks="inside", tickwidth=2, tickcolor='black', ticklen=8
    ),

    # Configure Y-Axis (Elevation) - Millimeter Graph Paper Look
    yaxis=dict(
        showgrid=True,
        gridcolor='rgb(160, 190, 160)', # Major grid: drafting green
        gridwidth=1.5,
        dtick=2,
        minor=dict(
            dtick=0.5,
            showgrid=True,
            gridcolor='rgb(210, 230, 210)', # Minor grid: faint green
            gridwidth=0.5
        ),
        zeroline=True,
        zerolinecolor='black',
        zerolinewidth=2,
        showline=True, linewidth=2, linecolor='black', mirror=True,
        ticks="inside", tickwidth=2, tickcolor='black', ticklen=8,
        range=[486, 510] # Pad the bottom slightly below river bed for a clean look
    ),

    # Add a border block around the graph
    shapes=[
        dict(
            type="rect",
            xref="paper", yref="paper",
            x0=0, y0=0, x1=1, y1=1,
            line=dict(color="black", width=3)
        )
    ],
    showlegend=False,
    margin=dict(l=80, r=40, t=80, b=60)
)

fig.show()

# Save the figure as an HTML file
fig.write_html("jawalgaon_capacity_curve.html")

In [7]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd
import numpy as np

# 1. Load the Datasets
df = pd.read_excel("Content Table.xlsx", sheet_name="Content TABLE ")
df_dam = pd.read_excel("Content Table.xlsx", sheet_name="DAM Station ELEV(GIS)")

# 2. Extract Data
elevation = df['Elevation (m)'].values
volume = df['Volume (MCM)'].values

# 3. Calculate Surface Area (in sq.km or Million m^2) mathematically: A = dV / dh
area = np.zeros_like(volume)
for i in range(1, len(elevation)):
    dh = elevation[i] - elevation[i-1]
    dv = volume[i] - volume[i-1]
    if dh > 0:
        area[i] = dv / dh
    else:
        area[i] = area[i-1]
area[0] = 0 # Base of river bed

# 4. Create Figure with Dual X-Axes (Top and Bottom)
fig = make_subplots(specs=[[{"secondary_y": False}]])

# Add the Capacity Curve (Volume) - Bottom Axis
fig.add_trace(go.Scatter(
    x=volume,
    y=elevation,
    mode='lines', # Changed from 'lines+markers' to 'lines'
    line_shape='spline', # Added for smoother curves
    name='Capacity (MCM)',
    line=dict(color='black', width=3, dash='solid')
))

# Add the Area Curve (Waterspread) - Top Axis
fig.add_trace(go.Scatter(
    x=area,
    y=elevation,
    mode='lines', # Changed from 'lines+markers' to 'lines'
    line_shape='spline', # Added for smoother curves
    name='Surface Area (Sq.km)',
    xaxis='x2', # Link to secondary X-axis on top
    line=dict(color='navy', width=2.5, dash='dashdot')
))

# 5. Identify Critical Reservoir Levels
levels = []
for idx, row in df.dropna(subset=['Remarks']).iterrows():
    levels.append((row['Elevation (m)'], row['Remarks']))
tbl_elev = df_dam['Crest Elevation (m)'].max()
levels.append((tbl_elev, 'Top of Bund Level (TBL)'))

# Add CAD-style Annotations for Critical Levels
for el, name in levels:
    fig.add_hline(
        y=el, line_dash="dash", line_color="red", line_width=1.5,
        annotation_text=f" {name} [EL {el:.2f} m] ",
        annotation_position="bottom left",
        annotation_font=dict(family="Courier New, monospace", size=11, color="black"),
        annotation_bgcolor="white",
        annotation_bordercolor="black",
        annotation_borderwidth=1
    )

# 6. Apply "Engineering Graph Paper" Formatting with Dual Axes
fig.update_layout(
    title=dict(
        text='<b>JAWALGAON DAM: ELEVATION-AREA-CAPACITY CURVES</b><br><sup>DRAWING NO: 02 | SCALE: AS SHOWN</sup>',
        font=dict(family="Courier New, monospace", size=18, color="black"),
        x=0.5,
        xanchor='center'
    ),
    plot_bgcolor='rgb(255, 255, 255)', # Pure white paper background
    paper_bgcolor='rgb(245, 245, 245)', # Slight grey surround
    font=dict(family="Courier New, monospace", size=12, color="black"),

    # Configure Bottom X-Axis (Volume)
    xaxis=dict(
        title='<b>STORAGE VOLUME (MCM)</b>',
        showgrid=True, gridcolor='rgb(160, 190, 160)', gridwidth=1.5, dtick=10,
        minor=dict(dtick=2, showgrid=True, gridcolor='rgb(210, 230, 210)', gridwidth=0.5),
        zeroline=True, zerolinecolor='black', zerolinewidth=2,
        showline=True, linewidth=2, linecolor='black', mirror=False,
        ticks="inside", tickwidth=2, tickcolor='black', ticklen=8
    ),

    # Configure Top X-Axis (Area)
    xaxis2=dict(
        title='<b>WATERSPREAD AREA (Sq.km)</b>',
        titlefont=dict(color='navy'),
        tickfont=dict(color='navy'),
        overlaying='x', side='top', # Position at the top
        showgrid=False, # Avoid overlapping vertical grids
        showline=True, linewidth=2, linecolor='black',
        ticks="inside", tickwidth=2, tickcolor='navy', ticklen=8,
        autorange='reversed' # Reverse this axis to create the desired visual effect
    ),

    # Configure Y-Axis (Elevation)
    yaxis=dict(
        title='<b>ELEVATION (m)</b>',
        showgrid=True, gridcolor='rgb(160, 190, 160)', gridwidth=1.5, dtick=2,
        minor=dict(dtick=0.5, showgrid=True, gridcolor='rgb(210, 230, 210)', gridwidth=0.5),
        zeroline=True, zerolinecolor='black', zerolinewidth=2,
        showline=True, linewidth=2, linecolor='black', mirror=True,
        ticks="inside", tickwidth=2, tickcolor='black', ticklen=8
    ),

    # Border block and Legend
    shapes=[dict(type="rect", xref="paper", yref="paper", x0=0, y0=0, x1=1, y1=1, line=dict(color="black", width=3))],
    showlegend=True,
    legend=dict(x=0.8, y=0.05, bordercolor="black", borderwidth=1, bgcolor="white"),
    margin=dict(l=80, r=40, t=100, b=60)
)

fig.show()

# Save the figure as an HTML file
fig.write_html("jawalgaon_eac_curves.html")

In [8]:
# ════════════════════════════════════════════════════════════════════════
# CELL — Install Playwright and browser dependencies for HTML to PNG conversion
# ════════════════════════════════════════════════════════════════════════
import subprocess, sys

print('Installing system dependencies for Playwright...')
subprocess.run(['apt-get', 'update'], check=False)
# Install common dependencies for headless Chrome on Linux
subprocess.run(['apt-get', 'install', '-y', 'libatk-bridge2.0-0', 'libcups2', 'libxcomposite1', 'libxdamage1', 'libxrandr2', 'libgtk-3-0', 'libgdk-pixbuf2.0-0', 'libfontconfig1', 'libatk1.0-0', 'libatk-bridge2.0-0', 'libxi6', 'libxkbcommon0', 'libxcursor1', 'libxshmfence1', 'libgbm1'], check=False)

print('Installing playwright...')
subprocess.run([sys.executable, '-m', 'pip', 'install', 'playwright', '--quiet'], check=False)
print('Installing browser drivers...')
# Install Chromium browser, as it's typically reliable for rendering
subprocess.run([sys.executable, '-m', 'playwright', 'install', 'chromium'], check=False)
print('✔  Playwright and browser drivers ready')


Installing system dependencies for Playwright...
Installing playwright...
Installing browser drivers...
✔  Playwright and browser drivers ready


In [9]:
# ════════════════════════════════════════════════════════════════════════
# CELL — Function to convert HTML to PNG using Playwright
# ════════════════════════════════════════════════════════════════════════
from playwright.async_api import async_playwright
import os
import asyncio # Import asyncio

async def html_to_png(html_file_path, output_dir='.', max_width=1600, max_height=None):
    """Converts an HTML file to a PNG image using Playwright.

    Args:
        html_file_path (str): Path to the input HTML file.
        output_dir (str): Directory to save the PNG file. Defaults to current directory.
        max_width (int): Maximum width for the screenshot. Defaults to 1600 pixels.
        max_height (int): Maximum height for the screenshot. If None, it will capture full scrollable height.
    """
    png_file_name = os.path.basename(html_file_path).replace('.html', '.png')
    output_path = os.path.join(output_dir, png_file_name)

    print(f'Converting {html_file_path} to {output_path}...')

    async with async_playwright() as p:
        browser = await p.chromium.launch()
        page = await browser.new_page()
        await page.goto(f'file://{os.path.abspath(html_file_path)}', wait_until='networkidle')

        # Get the scrollable height of the page content
        if max_height is None:
            # Evaluate in browser context to get the full scroll height
            page_height = await page.evaluate('document.body.scrollHeight')
        else:
            page_height = max_height

        # Set viewport to capture full height or specified height
        await page.set_viewport_size({'width': max_width, 'height': page_height})

        await page.screenshot(path=output_path, full_page=True)
        await browser.close()
    print(f'✔  Successfully converted {html_file_path} to {png_file_name}')


# List of HTML files to convert (from previous execution)
html_output_files = [
    "jawalgaon_capacity_curve.html",
    "jawalgaon_eac_curves.html"
]

# Convert each HTML file to PNG
# Run the async function using asyncio.run or await it in an async cell
async def convert_all_html_to_png():
    for html_file in html_output_files:
        if os.path.exists(html_file):
            await html_to_png(html_file)
        else:
            print(f'Warning: HTML file not found: {html_file}. Skipping conversion.')

    print('\nConversion of all available HTML graphs to PNG complete.')

# Execute the async function
await convert_all_html_to_png()

Converting jawalgaon_capacity_curve.html to ./jawalgaon_capacity_curve.png...
✔  Successfully converted jawalgaon_capacity_curve.html to jawalgaon_capacity_curve.png
Converting jawalgaon_eac_curves.html to ./jawalgaon_eac_curves.png...
✔  Successfully converted jawalgaon_eac_curves.html to jawalgaon_eac_curves.png

Conversion of all available HTML graphs to PNG complete.


In [10]:
# ════════════════════════════════════════════════════════════════════════
# CELL — Create a zip file of all generated PNG images
# ════════════════════════════════════════════════════════════════════════
import zipfile
import os

png_output_files = [f.replace('.html', '.png') for f in html_output_files]
zip_png_filename = 'Jawalgaon_DBA_Graphs_PNG.zip'

with zipfile.ZipFile(zip_png_filename, 'w') as zipf:
    for file in png_output_files:
        if os.path.exists(file):
            zipf.write(file)
            print(f'Added {file} to {zip_png_filename}')
        else:
            print(f'Warning: {file} not found and was not added to the zip.')

print(f'\nSuccessfully created {zip_png_filename} containing all generated PNG files.')

Added jawalgaon_capacity_curve.png to Jawalgaon_DBA_Graphs_PNG.zip
Added jawalgaon_eac_curves.png to Jawalgaon_DBA_Graphs_PNG.zip

Successfully created Jawalgaon_DBA_Graphs_PNG.zip containing all generated PNG files.
